# 6. Two-Optimizer Adversarial cVAE + MMD Batch Correction

**Pipeline:**
1. Load cohort from `data/interim_PT/`
2. Generate dummy split (`Split_dummy`: 60/20/20)
3. Fit Adv2-cVAE on **train** embeddings only
4. Transform both train and test -> 128-dim batch-invariant latent
5. Save to `data/processed_two_opt/`
6. Compare with KL-cVAE results

## 0. Environment Setup

In [ ]:
# 1) Clone / pull the repo
!git clone -b feat/scvi-batch-correction https://github.com/onion-42/batchcor-rna-embeds.git 2>/dev/null || (cd batchcor-rna-embeds && git pull origin feat/scvi-batch-correction)
%cd batchcor-rna-embeds

# 2) Install only packages Colab doesn't have (use Colab's native anndata/zarr/scanpy)
!pip install -q loguru tqdm umap-learn scib-metrics leidenalg igraph harmonypy

# 3) Add project root to Python path (avoids pip install -e . which pulls immuno-compass)
import sys
sys.path.insert(0, "/content/batchcor-rna-embeds")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from pathlib import Path
from loguru import logger

from batchcor_rna_emb.logging_config import set_logger
from batchcor_rna_emb.config import (
    BATCH_COL, DIAGNOSIS_COL, SEED,
    COMPASS_PT_EMBEDDING_KEY,
    SCVI_CORRECTED_KEY, SCVI_LATENT_DIM,
    COHORT_CANCER_CODE,
)
from batchcor_rna_emb.data_io import load_cohort, save_adata_zarr
from batchcor_rna_emb.split_utils import add_dummy_split, get_split_masks
from batchcor_rna_emb.batch_correction.scvi_corrector import (
    CVAEAdv2Config, CVAEAdv2Corrector,
)
from batchcor_rna_emb.metrics.batch_metrics import (
    compute_kbet, compute_ilisi, compute_asw_batch, compute_graph_connectivity,
)
from batchcor_rna_emb.metrics.bio_metrics import (
    compute_clisi, compute_silhouette_bio, compute_nmi, compute_ari,
    run_leiden_clustering,
)
from batchcor_rna_emb.metrics.aggregation import geometric_mean

set_logger(level="INFO")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
logger.info("Imports OK, device: {}", DEVICE)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATA_INTERIM_PT_DIR = Path('/content/drive/MyDrive/data/interim_PT')
DATA_PROCESSED_ADV  = Path('/content/drive/MyDrive/data/processed_two_opt')
DATA_PROCESSED_ADV.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED_DIR  = Path('/content/drive/MyDrive/data/processed')

SPLIT_COL = "Split_dummy"
ADV2_CORRECTED_KEY = COMPASS_PT_EMBEDDING_KEY + "_Adv2_corrected"
MULTI_BATCH_COHORTS = ["PUB_KIRC_ICI_combined", "NSCLC_Tissue_ICI_Pred"]

logger.info("Paths configured")
logger.info("Source: {}", DATA_INTERIM_PT_DIR)
logger.info("Dest:   {}", DATA_PROCESSED_ADV)

## 2. Load, Correct & Save

In [ ]:
results = {}

for cohort_name in MULTI_BATCH_COHORTS:
    logger.info("\n" + "=" * 70)
    logger.info("Processing: {}", cohort_name)
    logger.info("=" * 70)

    src_path = DATA_INTERIM_PT_DIR / f"{cohort_name}.adata.zarr"
    adata = load_cohort(src_path)
    logger.info("Loaded: {} samples x {} genes", adata.n_obs, adata.n_vars)

    add_dummy_split(adata, seed=SEED)
    train_mask, test_mask = get_split_masks(adata, SPLIT_COL)
    logger.info("Split: {} train, {} test, {} NaN",
                train_mask.sum(), test_mask.sum(),
                (~train_mask & ~test_mask).sum())

    X_all = adata.obsm[COMPASS_PT_EMBEDDING_KEY].astype(np.float32)
    batch_all = adata.obs[BATCH_COL].values

    n_batches = len(adata.obs[BATCH_COL].cat.categories)
    cfg = CVAEAdv2Config(
        latent_dim=SCVI_LATENT_DIM,
        n_epochs=300,
        normalize=True,
    )
    logger.info("Config: latent_dim={}, n_batches={}, n_epochs={}",
                cfg.latent_dim, n_batches, cfg.n_epochs)

    corrector = CVAEAdv2Corrector(cfg, device=DEVICE)
    history = corrector.fit(X_all[train_mask], batch_all[train_mask])

    Z_corrected = corrector.transform(X_all, batch_all)
    adata.obsm[ADV2_CORRECTED_KEY] = Z_corrected
    logger.info("Corrected embedding shape: {}", Z_corrected.shape)

    dst_path = DATA_PROCESSED_ADV / f"{cohort_name}.adata.zarr"
    save_adata_zarr(adata, dst_path)
    logger.info("Saved to: {}", dst_path)

    results[cohort_name] = {"adata": adata, "history": history}

logger.info("\nAll cohorts processed!")

## 3. Training Curves

In [ ]:
for cohort_name, res in results.items():
    history = res["history"]
    fig, axes = plt.subplots(1, 4, figsize=(20, 4))
    fig.suptitle(f"{cohort_name} — Training Curves", fontsize=14, fontweight="bold")

    axes[0].plot(history["recon_loss"], label="Recon (MSE)")
    axes[0].set_title("Reconstruction Loss")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")

    if "disc_acc" in history:
        axes[1].plot(history["disc_acc"], label="Disc Accuracy", color="orange")
        axes[1].axhline(1.0/res["adata"].obs[BATCH_COL].nunique(), ls="--", color="gray", label="Chance")
        axes[1].set_title("Discriminator Accuracy")
        axes[1].set_xlabel("Epoch"); axes[1].legend()

    if "adv_loss" in history:
        axes[2].plot(history["adv_loss"], label="Adversarial Loss", color="red")
        axes[2].set_title("Adversarial Loss (encoder)")
        axes[2].set_xlabel("Epoch")

    if "mmd_loss" in history:
        axes[3].plot(history["mmd_loss"], label="MMD Loss", color="green")
        axes[3].set_title("MMD Loss")
        axes[3].set_xlabel("Epoch")

    plt.tight_layout()
    plt.show()

## 4. UMAP Before vs After

In [ ]:
from umap import UMAP

for cohort_name, res in results.items():
    adata = res["adata"]
    X_before = adata.obsm[COMPASS_PT_EMBEDDING_KEY]
    X_after  = adata.obsm[ADV2_CORRECTED_KEY]

    umap_before = UMAP(n_components=2, random_state=SEED).fit_transform(X_before)
    umap_after  = UMAP(n_components=2, random_state=SEED).fit_transform(X_after)

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(f"{cohort_name} — UMAP Before vs After Adv2-cVAE", fontsize=14, fontweight="bold")

    batch_labels = adata.obs[BATCH_COL].values
    diag_labels  = adata.obs[DIAGNOSIS_COL].values if DIAGNOSIS_COL in adata.obs.columns else None

    for j, (emb, title) in enumerate([(umap_before, "Before"), (umap_after, "After Adv2")]):
        ax = axes[0, j]
        for b in sorted(set(batch_labels)):
            mask = batch_labels == b
            ax.scatter(emb[mask, 0], emb[mask, 1], s=8, alpha=0.5, label=b)
        ax.set_title(f"{title} — RNA Batch")
        ax.legend(fontsize=6, markerscale=2, loc="best")

        if diag_labels is not None:
            ax = axes[1, j]
            for d in sorted(set(diag_labels)):
                mask = diag_labels == d
                ax.scatter(emb[mask, 0], emb[mask, 1], s=8, alpha=0.5, label=d)
            ax.set_title(f"{title} — Diagnosis")
            ax.legend(fontsize=6, markerscale=2, loc="best")

    plt.tight_layout()
    plt.show()

## 5. Silhouette Score Summary

In [ ]:
from sklearn.metrics import silhouette_score

rows = []
for cohort_name, res in results.items():
    adata = res["adata"]
    batch = adata.obs[BATCH_COL].values
    diag  = adata.obs[DIAGNOSIS_COL].values if DIAGNOSIS_COL in adata.obs.columns else None

    for key, label in [(COMPASS_PT_EMBEDDING_KEY, "Raw PT"),
                       (ADV2_CORRECTED_KEY, "Adv2-cVAE")]:
        emb = adata.obsm[key]
        sil_b = silhouette_score(emb, batch) if len(set(batch)) > 1 else np.nan
        sil_d = silhouette_score(emb, diag) if diag is not None and len(set(diag)) > 1 else np.nan
        rows.append({"Cohort": cohort_name, "Method": label,
                     "Sil_Batch": round(sil_b, 4) if not np.isnan(sil_b) else np.nan,
                     "Sil_Diag": round(sil_d, 4) if not np.isnan(sil_d) else np.nan})

df_sil = pd.DataFrame(rows)
display(df_sil)

## 6. Comparison with KL-cVAE (Notebook 4)

In [ ]:
rows_cmp = []

for cohort_name in MULTI_BATCH_COHORTS:
    adv_adata = results[cohort_name]["adata"]
    batch = adv_adata.obs[BATCH_COL].values
    n_uniq = len(set(batch))

    emb_adv = adv_adata.obsm[ADV2_CORRECTED_KEY]
    sil_adv = silhouette_score(emb_adv, batch) if n_uniq > 1 else np.nan

    kl_path = DATA_PROCESSED_DIR / f"{cohort_name}.adata.zarr"
    sil_kl = np.nan
    if kl_path.exists():
        try:
            kl_adata = load_cohort(kl_path)
            if SCVI_CORRECTED_KEY in kl_adata.obsm:
                emb_kl = kl_adata.obsm[SCVI_CORRECTED_KEY]
                sil_kl = silhouette_score(emb_kl, kl_adata.obs[BATCH_COL].values) if n_uniq > 1 else np.nan
        except Exception as e:
            logger.warning("Could not load KL-cVAE result for {}: {}", cohort_name, e)

    emb_raw = adv_adata.obsm[COMPASS_PT_EMBEDDING_KEY]
    sil_raw = silhouette_score(emb_raw, batch) if n_uniq > 1 else np.nan

    rows_cmp.append({
        "Cohort": cohort_name,
        "Sil_Batch_Raw": round(sil_raw, 4),
        "Sil_Batch_KL_cVAE": round(sil_kl, 4) if not np.isnan(sil_kl) else np.nan,
        "Sil_Batch_Adv2": round(sil_adv, 4),
    })

df_cmp = pd.DataFrame(rows_cmp)
display(df_cmp)

## 7. Full scIB Metrics (KIRC + NSCLC)

In [ ]:
scib_rows = []

for cohort_name, res in results.items():
    adata = res["adata"]
    batch_col = BATCH_COL
    diag_col  = DIAGNOSIS_COL if DIAGNOSIS_COL in adata.obs.columns else None

    for key, method in [(COMPASS_PT_EMBEDDING_KEY, "Raw_PT"),
                        (ADV2_CORRECTED_KEY, "Adv2_cVAE")]:
        emb = adata.obsm[key]
        batch = adata.obs[batch_col]
        row = {"Cohort": cohort_name, "Method": method}

        try: row["kBET"] = round(compute_kbet(emb, batch), 4)
        except: row["kBET"] = np.nan
        try: row["iLISI"] = round(compute_ilisi(emb, batch), 4)
        except: row["iLISI"] = np.nan
        try: row["ASW_batch"] = round(compute_asw_batch(emb, batch), 4)
        except: row["ASW_batch"] = np.nan
        try: row["graph_conn"] = round(compute_graph_connectivity(emb, batch), 4)
        except: row["graph_conn"] = np.nan

        batch_vals = [v for v in [row.get("kBET"), row.get("iLISI"),
                                   row.get("ASW_batch"), row.get("graph_conn")] if not np.isnan(v)]
        row["AvgBatch"] = round(np.mean(batch_vals), 4) if batch_vals else np.nan

        if diag_col and len(adata.obs[diag_col].unique()) > 1:
            diag = adata.obs[diag_col]
            try: row["cLISI"] = round(compute_clisi(emb, diag), 4)
            except: row["cLISI"] = np.nan
            try: row["Sil_bio"] = round(compute_silhouette_bio(emb, diag), 4)
            except: row["Sil_bio"] = np.nan
            try:
                leiden = run_leiden_clustering(emb)
                row["NMI"] = round(compute_nmi(diag, leiden), 4)
                row["ARI"] = round(compute_ari(diag, leiden), 4)
            except:
                row["NMI"] = np.nan; row["ARI"] = np.nan
            bio_vals = [v for v in [row.get("cLISI"), row.get("Sil_bio"),
                                     row.get("NMI"), row.get("ARI")] if not np.isnan(v)]
            row["AvgBio"] = round(np.mean(bio_vals), 4) if bio_vals else np.nan
        else:
            row["cLISI"] = np.nan; row["Sil_bio"] = np.nan
            row["NMI"] = np.nan; row["ARI"] = np.nan; row["AvgBio"] = np.nan

        scib_rows.append(row)

df_scib = pd.DataFrame(scib_rows)
display(df_scib)